# Bước 2: Data Collection, Understanding & Preparation
## Kiểm Toán Chất Lượng Dữ Liệu Theo Khung DAMA-DMBOK & Lưu Vết (Audit Trail)
### Phân Tích Hành Vi Mua Hàng Của Khách Hàng Trong Thương Mại Điện Tử (Tập Dữ Liệu Gốc: 500.000 Bản Ghi)
#### Đề án Môn học: Applied Data Analytics (ADY201m) — ĐH FPT
---


## 2.0 Tuyên ngôn Các Nguyên tắc Quản trị Dữ liệu (Data Governance Principles)

Nhằm đảm bảo tính khoa học, minh bạch và khả năng tái lập trong thực tế doanh nghiệp, toàn bộ quy trình tiền xử lý dữ liệu của dự án được xây dựng dựa trên 3 trụ cột cốt lõi:

1. **Khung Đánh Giá Toàn Diện (DAMA-DMBOK 6 Chiều):**
   Đánh giá và kiểm toán chất lượng dữ liệu có hệ thống theo 6 chiều chuẩn mực quốc tế:
   - *Completeness (Đầy đủ):* Không chứa ô trống (Null/NaN) trên toàn bộ 500.000 bản ghi.
   - *Uniqueness (Duy nhất):* Khử trùng lặp (Deduplication), đảm bảo mỗi bản ghi là một phiên tương tác độc nhất.
   - *Validity (Hợp lệ):* Giá trị biến số phải nằm đúng miền logic thực tế (không chấp nhận số âm ở biến đếm hoặc biến thời gian).
   - *Consistency (Nhất quán):* Logic nghiệp vụ giữa các thuộc tính không mâu thuẫn (ví dụ: ngày mua gần nhất không thể lớn hơn thâm niên tài khoản).
   - *Accuracy (Chính xác):* Giữ nguyên độ chính xác số học (Round-trip precision), không làm tròn sai lệch kiểu số.
   - *Timeliness (Kịp thời):* Phản ánh đúng thời điểm phát sinh hành vi, kiểm soát chặt chẽ mốc thời gian để loại bỏ hoàn toàn rủi ro rò rỉ dữ liệu (No Data Leakage).

2. **Kỹ Thuật Lưu Vết Kiểm Toán (Audit Trail & Reproducibility):**
   - *Tính bất biến của dữ liệu gốc:* Tuyệt đối không can thiệp hay ghi đè lên file dữ liệu ban đầu. Kiểm định tính toàn vẹn cấu trúc (500.000 dòng × 17 cột) bằng bảng cờ lỗi `issues` độc lập.
   - *Phân tách có lưu vết:* Tách 100% bản ghi vi phạm (67.744 dòng) sang kho lưu trữ riêng (`customerData_rejected.csv`) theo đúng chỉ số dòng gốc, đảm bảo mọi thao tác đều có thể truy vết và kiểm chứng.

3. **Bảo Toàn Tính Đại Diện & Chân Thực Của Thị Trường:**
   - Phân định rạch ròi giữa **dữ liệu sai hỏng kỹ thuật/logic** (7 vấn đề cần bóc tách) và **biến thiên tự nhiên của khách hàng thật** (như nhóm khách hàng có thu nhập cao hoặc gắn bó lâu năm).
   - Dự án bảo toàn trọn vẹn các quan sát thực tế này, không sử dụng công thức toán cắt gọt cơ học (Trimming hay Winsorization theo IQR) làm méo mó chân dung khách hàng thật.

## 2.1 Import Thư viện & Thiết lập Môi trường Làm việc

**Mục tiêu:** Nạp các thư viện phân tích dữ liệu cốt lõi (`pandas`, `numpy`, `os`) và tự động thiết lập đường dẫn truy xuất tệp dữ liệu, đảm bảo code chạy tương thích và ổn định ở mọi môi trường thực thi.


In [22]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = '../Data'

RAW_PATH = f'{DATA_DIR}/customerData_500k.csv'
CLEAN_PATH = f'{DATA_DIR}/customerData_clean.csv'
REJECTED_PATH = f'{DATA_DIR}/customerData_rejected.csv'

## 2.2 Nạp Dữ Liệu & Khảo Sát Cấu Trúc Tổng Quan (Shape & Dtypes)

* **Mục tiêu:** Nạp tệp dữ liệu thô customerData_500k.csv, xác thực kích thước tổng thể (số dòng, số cột) và kiểm tra kiểu dữ liệu của 17 thuộc tính.

In [ ]:
raw0 = pd.read_csv(RAW_PATH)
print('Kích thước (dòng, cột):', raw0.shape)
print(raw0.dtypes)

Kích thước (dòng, cột): (500000, 17)
--- customerData ---
Age                       int64
AnnualIncome            float64
NumberOfPurchases         int64
TimeSpentOnWebsite      float64
CustomerTenureYears     float64
LastPurchaseDaysAgo       int64
Gender                      str
ProductCategory             str
PreferredDevice             str
Region                      str
ReferralSource              str
CustomerSegment             str
LoyaltyProgram            int64
DiscountsAvailed          int64
SessionCount              int64
CustomerSatisfaction    float64
PurchaseStatus            int64
dtype: object


**Kết luận:**
* Bảng dữ liệu thô gồm **500.000 dòng × 17 cột**.
* Phân bố kiểu dữ liệu: 8 thuộc tính số nguyên (int64), 3 thuộc tính số thực (

loat64) và 6 thuộc tính phân loại dạng chuỗi (object). Các thuộc tính đều có kiểu dữ liệu phù hợp với ý nghĩa nghiệp vụ ban đầu, chưa phát hiện trường hợp số bị lưu nhầm dạng text.

### 2.2.1 Kiểm Toán & Xử Lý Tính Đầy Đủ (Completeness - Missing Value Handling)

**Câu hỏi kiểm toán:** Trong 17 thuộc tính, những cột nào có giá trị bị khuyết thiếu và số lượng thiếu là bao nhiêu? Cần áp dụng chiến lược xử lý nào để bảo toàn cấu trúc dữ liệu mà không làm lệch phân phối?

In [ ]:
print("Số ô khuyết thiếu:")
print(raw0[['AnnualIncome', 'CustomerSatisfaction']].isnull().sum())

raw0[['AnnualIncome', 'CustomerSatisfaction']].describe().round(2)

Số ô khuyết thiếu:
AnnualIncome            987
CustomerSatisfaction    575
dtype: int64


,AnnualIncome,CustomerSatisfaction
count,499013.00,499425.00
mean,85068.57,3.22
std,39589.57,0.83
min,11966.39,1.00
25%,51991.31,3.00
50%,83750.20,3.00
75%,116550.05,4.00
max,204178.29,5.00


**Căn cứ lựa chọn phương pháp xử lý dữ liệu khuyết thiếu:**

* **Không chọn xóa dòng (Listwise Deletion):**
  Tỷ lệ thiếu rất nhỏ: `AnnualIncome` thiếu 987 ô (0,20%) và `CustomerSatisfaction` thiếu 575 ô (0,12%) — tổng cộng 1.562 ô (0,31% toàn bảng). Việc xóa dòng sẽ làm mất mát dữ liệu không cần thiết của 15 thuộc tính còn lại, do đó điền khuyết là giải pháp tối ưu để bảo toàn trọn vẹn 500.000 bản ghi.

* **Cân nhắc lựa chọn giữa Trung bình (Mean) và Trung vị (Median):**
  Dựa vào bảng `describe()` ở trên:
  * **Đối với Thu nhập (`AnnualIncome`):** 
    * Mean $\approx 85.069$ USD và Median (hàng 50%) $\approx 83.750$ USD xấp xỉ nhau (chênh lệch chỉ ~1,5%).
    * **Chọn Median** theo thông lệ chuẩn an toàn của dữ liệu thu nhập để hạn chế ảnh hưởng của nhóm khách hàng thu nhập cao ở cận trên.
  * **Đối với Điểm đánh giá (`CustomerSatisfaction`):** 
    * Thang đo 1–5 sao chỉ nhận giá trị nguyên. Mean $= 3,22$ sao bị lẻ không có thực trên phiếu khảo sát.
    * **Chọn Median (hàng 50%) = 3,0 sao** để giữ đúng thang điểm nguyên và là mức trung tính khách quan khi khách hàng không đánh giá.

In [ ]:
med_income = raw0['AnnualIncome'].median()
med_sat = float(raw0['CustomerSatisfaction'].median())

raw0['AnnualIncome'] = raw0['AnnualIncome'].fillna(med_income)
raw0['CustomerSatisfaction'] = raw0['CustomerSatisfaction'].fillna(med_sat)
tables['customerData'] = raw0

print("Xử lý hoàn tất:")
print(f"- Điền trung vị AnnualIncome: {med_income:,.0f} USD")
print(f"- Điền trung vị CustomerSatisfaction: {med_sat:.1f} sao")
print(f"- Kiểm tra lại số ô thiếu toàn bảng: {raw0.isnull().sum().sum()} ô")

Xử lý hoàn tất:
- Điền trung vị AnnualIncome: 83,750 USD
- Điền trung vị CustomerSatisfaction: 3.0 sao
- Kiểm tra lại số ô thiếu toàn bảng: 0 ô


**Kết luận:** Quá trình xử lý dữ liệu khuyết thiếu hoàn tất. Dữ liệu đạt tính đầy đủ (Completeness) 100% với 0 ô khuyết thiếu, bảo toàn trọn vẹn 500.000 bản ghi trước khi tiến hành các bước thăm dò phân phối và kiểm toán nghiệp vụ tiếp theo.

### 2.2.2 Khảo Sát Ngoại Lai Bằng Phương Pháp IQR (Tukey Outliers)

* **Mục tiêu:** Sử dụng rào trên Tukey ($\text{Rào trên} = Q3 + 1.5 \times \text{IQR}$) để kiểm tra ngoại lai trên 6 biến định lượng và quyết định phương án xử lý phù hợp.
* **6 biến số:** `Age`, `AnnualIncome`, `NumberOfPurchases`, `TimeSpentOnWebsite`, `CustomerTenureYears`, `LastPurchaseDaysAgo`.

In [ ]:
cols = ['Age', 'AnnualIncome', 'NumberOfPurchases', 'TimeSpentOnWebsite', 'CustomerTenureYears', 'LastPurchaseDaysAgo']
q1, q3 = raw0[cols].quantile(0.25), raw0[cols].quantile(0.75)
rao_tren = (q3 + 1.5 * (q3 - q1)).round(2)

pd.DataFrame({
    'Rao_tren_IQR': rao_tren,
    'Max_thuc_te': raw0[cols].max().round(2),
    'So_luong_vuot': (raw0[cols] > rao_tren).sum(),
    'Ty_le_%': ((raw0[cols] > rao_tren).mean() * 100).round(2)
})

,Rao_tren_IQR,Max_thuc_te,So_luong_vuot,Ty_le_%
Age,97.50,81.00,0,0.00
AnnualIncome,213124.80,204178.29,0,0.00
NumberOfPurchases,31.00,28.00,0,0.00
TimeSpentOnWebsite,88.77,78.36,0,0.00
CustomerTenureYears,6.64,15.35,24740,4.95
LastPurchaseDaysAgo,238.50,189.00,0,0.00


**Nhận xét & Quyết định xử lý ngoại lai:**

* **5 biến không có ngoại lai (0 bản ghi vượt rào):** `Age` (max 81 tuổi), `AnnualIncome` (max 204k USD), `NumberOfPurchases` (max 28 lần), `TimeSpentOnWebsite` (max 78.4 phút), `LastPurchaseDaysAgo` (max 189 ngày) đều có giá trị lớn nhất nằm an toàn dưới rào trên IQR, phản ánh đúng hành vi tiêu dùng thực tế.
* **Biến thâm niên (`CustomerTenureYears`) có ngoại lai:** 24.740 khách hàng (chiếm 4,95%) có thâm niên từ 6.7 đến 15.4 năm (vượt rào trên 6.64 năm).
* **Quyết định: KHÔNG XÓA nhóm vượt rào này.** Thâm niên lâu năm là khách hàng trung thành thực tế của sàn, nếu máy móc dùng IQR cắt bỏ sẽ loại bỏ nhầm nhóm khách hàng giá trị cao. Các lỗi logic thực sự (như tuổi mở tài khoản < 13) sẽ được xử lý bằng 7 quy tắc nghiệp vụ ở Mục 2.3.

### 2.2.3 Kiểm Toán & Xử Lý Dòng Trùng Lặp (Uniqueness)

**Câu hỏi kiểm toán:** Bộ dữ liệu có tồn tại các bản ghi trùng lặp hoàn toàn trên cả 17 thuộc tính không? Nếu có, đây là lỗi hệ thống nhân đôi bản ghi (Duplicate Logging) hay sự trùng hợp ngẫu nhiên giữa hai khách hàng? Việc loại bỏ có làm thay đổi phân phối dữ liệu hay không?

In [ ]:
print(f"Số dòng trùng lặp phát hiện: {raw0.duplicated().sum():,} dòng ({raw0.duplicated().mean()*100:.2f}%)")
print(f"Số lần lặp của các bản ghi trùng: {raw0[raw0.duplicated(keep=False)].value_counts().unique()}")

raw = raw0.drop_duplicates().reset_index(drop=True)
tables['customerData'] = raw

eval_cols = ['Age', 'AnnualIncome', 'PurchaseStatus']
cmp = pd.DataFrame({'Truoc': raw0[eval_cols].mean(), 'Sau': raw[eval_cols].mean()})
cmp['Lech_%'] = ((cmp['Sau'] / cmp['Truoc'] - 1) * 100).round(4)
cmp.round(4)

Số dòng trùng lặp phát hiện: 1,148 dòng (0.23%)
Số lần lặp của các bản ghi trùng: [2]


,Truoc,Sau,Lech_%
Age,43.9416,43.9450,0.0079
AnnualIncome,85065.9653,85067.5793,0.0019
PurchaseStatus,0.4183,0.4186,0.0648


**Nhận xét & Quyết định xử lý:**

* **Xác định bản chất:** Phát hiện **1.148 bản ghi trùng lặp hoàn toàn** (chiếm 0,23% tổng số dòng), mỗi bản ghi trùng đều xuất hiện đúng 2 lần (`[2]`). Do dữ liệu gồm 17 thuộc tính với nhiều biến số thực lẻ (thu nhập, thâm niên, thời gian web đến từng chữ số thập phân), việc hai khách hàng ngoài đời ngẫu nhiên trùng khớp 100% là bất khả thi. Đây chắc chắn là lỗi kỹ thuật nhân đôi bản ghi của hệ thống (Duplicate Logging).
* **Mức độ tác động thống kê cực tiểu:** Bảng so sánh trên cho thấy độ lệch trước và sau khử trùng là không đáng kể (Tuổi và Thu nhập lệch $< 0,01\%$, tỷ lệ mua hàng `PurchaseStatus` chỉ lệch $0,06\%$, hoàn toàn không làm méo mó phân phối dữ liệu).
* **Quyết định: Loại bỏ 1.148 bản sao bằng `drop_duplicates()`, chỉ giữ lại bản ghi đầu tiên.** Bộ dữ liệu sẵn sàng với **498.852 bản ghi duy nhất** để bước vào kiểm toán logic nghiệp vụ ở Mục 2.3.

## 2.3 Thiết Lập Bảng Lưu Vết Kiểm Toán (Audit Trail) Cho 7 Vấn Đề Nghiệp Vụ

Tuân thủ nguyên tắc quản trị dữ liệu, quy trình làm sạch **không xóa trực tiếp dữ liệu thô** mà sử dụng cơ chế **Lưu vết kiểm toán (Audit Trail)**. Bảng `issues` được khởi tạo song song với 498.852 bản ghi sau khử trùng lặp để ghi nhận độc lập các cờ vi phạm (flags) qua 7 quy tắc nghiệp vụ, giúp việc phân tách dữ liệu minh bạch và truy xuất được nguồn gốc lỗi.

In [29]:
# Khởi tạo bảng lưu vết kiểm toán lỗi cho 500.000 bản ghi gốc
issues = pd.DataFrame(index=raw.index)

### 2.3.1 Vấn đề 1: Sai miền giá trị (Số âm đơn biến)

* **Thuộc tính & Đơn vị sử dụng:**
  * `NumberOfPurchases`: Số lần mua hàng tích lũy trong lịch sử (lần, số nguyên).
  * `TimeSpentOnWebsite`: Thời gian truy cập trong phiên (phút, số thực).
  * `CustomerTenureYears`: Thâm niên tài khoản (năm, số thực).
  * `LastPurchaseDaysAgo`: Số ngày kể từ lần mua gần nhất (ngày, số nguyên).
* **Công thức toán học & Mâu thuẫn logic:**
  * Theo tiên đề toán học và thực tế vật lý, thời gian và tần suất hành động của người dùng bắt buộc phải không âm: $\text{Value} \ge 0$.
  * Điều kiện vi phạm: $\text{Value} < 0$.
* **Hiện trạng thực nghiệm trên dữ liệu:**
  * Phát hiện **27.435 bản ghi mang giá trị âm (chiếm 5,50% dữ liệu)**.
  * Bản chất: Giá trị `-1` ở số lần mua là mã lỗi kỹ thuật (Error Code) trả về từ cổng thanh toán khi giao dịch bị gián đoạn; trong khi thời gian âm phát sinh do lệch xung nhịp đồng hồ (Clock Skew) giữa các máy chủ ghi log.
* **Phân loại quy tắc & Căn cứ xử lý:**
  * *Loại quy tắc:* **Bắt buộc (Constraint Rule)** — Ràng buộc miền giá trị toán học cơ bản.
  * *Biến mục tiêu:* **Không xem `PurchaseStatus`** (hoàn toàn khách quan).
  * *Căn cứ xử lý:* Tuyệt đối không dùng hàm trị tuyệt đối `abs()` vì sẽ biến mã lỗi `-1` thành khách mua hàng 1 lần thật; cũng không thể điền số trung bình để hợp thức hóa dữ liệu hỏng. Bóc tách toàn bộ 27.435 dòng này vào tệp kiểm toán `customerData_rejected.csv` để lưu vết.

In [30]:
# Vấn đề 1: Đánh dấu các bản ghi có giá trị âm ở 4 biến định lượng
issues['neg_values'] = (raw[['NumberOfPurchases', 'TimeSpentOnWebsite', 'CustomerTenureYears', 'LastPurchaseDaysAgo']] < 0).any(axis=1)

### 2.3.2 Vấn đề 2: Nghịch lý thời gian — Mua hàng trước khi tạo tài khoản

* **Thuộc tính & Đơn vị sử dụng:**
  * `CustomerTenureYears`: Thâm niên tài khoản tính bằng năm (thời gian từ lúc đăng ký tài khoản đến mốc chốt dữ liệu).
  * `LastPurchaseDaysAgo`: Số ngày tính từ lần mua gần nhất đến mốc chốt dữ liệu.
* **Công thức toán học & Mâu thuẫn logic:**
  * Tổng số ngày tối đa mà một tài khoản tồn tại được quy đổi theo công thức: 
    $$T_{\text{max}} = \text{CustomerTenureYears} \times 365.25 \text{ (ngày)}$$
  * Điều kiện vi phạm trật tự thời gian: 
    $$\text{LastPurchaseDaysAgo} > T_{\text{max}}$$
  * *Ý nghĩa logic:* Khoảng cách từ lần mua gần nhất đến nay lại **lớn hơn tổng số ngày tài khoản được tạo lập**. Điều này đồng nghĩa với việc: **Giao dịch mua hàng diễn ra trước cả ngày khách hàng đăng ký tài khoản!**
* **Hiện trạng thực nghiệm trên dữ liệu:**
  * Phát hiện **40.666 bản ghi vi phạm (chiếm 8,15% dữ liệu)**.
  * *Dẫn chứng cụ thể:* Có tài khoản mới mở được khoảng 7 ngày (`CustomerTenureYears = 0.02` năm), nhưng hệ thống lại ghi nhận phát sinh đơn mua từ 180 ngày trước (`LastPurchaseDaysAgo = 180`), tức mua hàng trước ngày mở nick tới 173 ngày. Lỗi này xuất phát từ việc gộp lịch sử mua của khách vãng lai (Guest Checkout) vào tài khoản mới mà không đồng bộ lại mốc thời gian đăng ký.
* **Phân loại quy tắc & Căn cứ xử lý:**
  * *Loại quy tắc:* **Bắt buộc (Constraint Rule)** — Ràng buộc tính toàn vẹn thời gian (Temporal Integrity).
  * *Biến mục tiêu:* **Không xem `PurchaseStatus`**.
  * *Căn cứ xử lý:* Dữ liệu hiện tại không có bảng nhật ký giao dịch chi tiết để xác định mốc thời gian nào đúng. Việc tự ý lùi ngày tạo nick hoặc đặt lại ngày mua đều là can thiệp chủ quan, tạo ra dữ liệu sai lệch. Do đó, bóc tách toàn bộ 40.666 bản ghi này vào `customerData_rejected.csv` là phương án minh bạch duy nhất.

In [31]:
# Vấn đề 2: Mua hàng trước ngày tạo tài khoản
issues['purchase_before_tenure'] = raw['LastPurchaseDaysAgo'] > (raw['CustomerTenureYears'] * 365.25)

### 2.3.3 Vấn đề 3: Mâu thuẫn giao dịch — Chưa mua hàng nhưng có ngày mua gần nhất

* **Thuộc tính & Đơn vị sử dụng:**
  * `NumberOfPurchases`: Tổng số lần mua hàng tích lũy trong lịch sử tài khoản (lần).
  * `LastPurchaseDaysAgo`: Số ngày tính từ lần mua gần nhất đến hiện tại (ngày).
* **Công thức toán học & Mâu thuẫn logic:**
  * Điều kiện kiểm toán vi phạm: $\text{NumberOfPurchases} = 0 \text{ và } \text{LastPurchaseDaysAgo} > 0$.
  * *Mâu thuẫn logic:* Nếu một khách hàng chưa từng phát sinh bất kỳ đơn hàng nào trong lịch sử ($N = 0$), thì thuộc tính "ngày mua gần nhất" bắt buộc phải bằng 0 (hoặc quy ước không xác định). Việc tồn tại một số ngày cụ thể (ví dụ: đã mua cách đây 45 ngày) khi tổng số lần mua bằng 0 là sự xung đột trực tiếp giữa biến đếm tích lũy và biến thời gian sự kiện.
* **Hiện trạng thực nghiệm trên dữ liệu:**
  * Phát hiện **8.125 bản ghi mâu thuẫn (chiếm 1,63% dữ liệu)**. Bản ghi bị xung đột nội tại: một thuộc tính khẳng định "chưa từng mua", thuộc tính khác lại khẳng định "đã mua cách đây X ngày".
* **Phân loại quy tắc & Căn cứ xử lý:**
  * *Loại quy tắc:* **Bắt buộc (Constraint Rule)** — Tính nhất quán nội tại giữa các trường dữ liệu.
  * *Biến mục tiêu:* **Không xem `PurchaseStatus`**.
  * *Căn cứ xử lý:* Không thể phỏng đoán cột nào ghi nhận đúng để sửa, giải pháp an toàn là bóc tách vào kho kiểm toán `customerData_rejected.csv`.

In [32]:
# Vấn đề 3: Chưa mua hàng nhưng có ngày mua gần nhất
issues['zero_purchases_has_last_date'] = (raw['NumberOfPurchases'] == 0) & (raw['LastPurchaseDaysAgo'] > 0)

### 2.3.4 Vấn đề 4: Mâu thuẫn khuyến mãi — Chưa mua hàng nhưng đã tiêu dùng mã giảm giá

* **Thuộc tính & Đơn vị sử dụng:**
  * `NumberOfPurchases`: Tổng số lần mua hàng tích lũy (lần).
  * `DiscountsAvailed`: Số lượng mã giảm giá **đã được sử dụng thành công trong đơn hàng** (mã).
* **Công thức toán học & Mâu thuẫn logic:**
  * Điều kiện kiểm toán vi phạm: $\text{NumberOfPurchases} = 0 \text{ và } \text{DiscountsAvailed} > 0$.
  * *Mâu thuẫn logic:* Thuộc tính `DiscountsAvailed` (khác với voucher được lưu trong ví) ghi nhận số lượng voucher đã được khấu trừ vào các giao dịch thanh toán thành công. Một khách hàng có 0 đơn hàng thành công thì không thể có lịch sử tiêu dùng thành công từ 1 đến nhiều mã giảm giá.
* **Hiện trạng thực nghiệm trên dữ liệu:**
  * Phát hiện **7.530 bản ghi vi phạm (chiếm 1,51% dữ liệu)**. Đây là lỗi đồng bộ từ dịch vụ quản lý khuyến mãi khi ghi nhận cả các voucher đã bấm áp dụng thử hoặc voucher đã hết hạn vào chỉ số tiêu dùng thực tế.
* **Phân loại quy tắc & Căn cứ xử lý:**
  * *Loại quy tắc:* **Bắt buộc (Constraint Rule)** — Tính nhất quán nghiệp vụ thương mại điện tử.
  * *Biến mục tiêu:* **Không xem `PurchaseStatus`**.
  * *Căn cứ xử lý:* Bản ghi phản ánh sai lệch hành vi tiêu dùng khuyến mãi, bóc tách sang `customerData_rejected.csv`.

In [33]:
# Vấn đề 4: Chưa mua hàng nhưng có lượt áp dụng mã giảm giá
issues['zero_purchases_used_discounts'] = (raw['NumberOfPurchases'] == 0) & (raw['DiscountsAvailed'] > 0)

### 2.3.5 Vấn đề 5: Mâu thuẫn phân khúc — Chưa từng mua hàng nhưng xếp hạng VIP/Premium

* **Thuộc tính & Đơn vị sử dụng:**
  * `NumberOfPurchases`: Tổng số lần mua hàng tích lũy trong lịch sử (lần).
  * `CustomerSegment`: Phân hạng khách hàng trong hệ thống CRM (`Regular`, `Premium`, `VIP`).
* **Công thức toán học & Mâu thuẫn logic:**
  * Điều kiện kiểm toán vi phạm: $\text{NumberOfPurchases} = 0 \text{ và } \text{CustomerSegment} \in [\text{'VIP'}, \text{'Premium'}]$.
  * *Làm rõ về nghiệp vụ:* Khách hàng VIP/Premium có thể mua ít lần (ví dụ: chỉ mua 1–2 lần nhưng giá trị đơn hàng rất lớn), điều đó hoàn toàn hợp lệ. Tuy nhiên, mâu thuẫn ở đây nằm ở con số **$0$ tuyệt đối**: Một tài khoản mới đăng ký, **chưa từng phát sinh bất kỳ một giao dịch nào trong đời ($N=0$)**, không thể được hệ thống phân hạng vào tệp khách hàng cao cấp hoặc đặc quyền ngay từ đầu.
  * Phân khúc ban đầu của một người dùng chưa mua sắm bắt buộc phải là phân khúc phổ thông (`Regular` / New User). Việc tài khoản $N=0$ mang nhãn VIP/Premium phản ánh **lỗi gán nhãn phân khúc mặc định (Default Classification Error)** hoặc lỗi đồng bộ sai lệch từ hệ thống CRM.
* **Hiện trạng thực nghiệm trên dữ liệu:**
  * Phát hiện **6.422 bản ghi vi phạm (chiếm 1,29% dữ liệu)**.
* **Phân loại quy tắc & Căn cứ xử lý:**
  * *Loại quy tắc:* **Quyết định (Decision Rule)** — Dựa trên nguyên tắc quản trị phân khúc khách hàng của doanh nghiệp.
  * *Biến mục tiêu:* **Không xem `PurchaseStatus`**.
  * *Căn cứ xử lý:* Để bảo đảm tệp khách hàng cao cấp phản ánh đúng những người có giá trị đóng góp thực tế, nhóm quyết định bóc tách nhóm vi phạm này sang `customerData_rejected.csv` để rà soát quy trình cấp thẻ thành viên.

In [34]:
# Vấn đề 5: Chưa mua hàng nhưng xếp hạng VIP/Premium
issues['zero_purchases_vip_segment'] = (raw['NumberOfPurchases'] == 0) & (raw['CustomerSegment'].isin(['VIP', 'Premium']))

### 2.3.6 Vấn đề 6: Vi phạm tính hợp lý dân sự — Tuổi tạo tài khoản dưới 13 tuổi

* **Thuộc tính & Đơn vị sử dụng:**
  * `Age`: Tuổi hiện tại của khách hàng (tuổi, số nguyên).
  * `CustomerTenureYears`: Số năm thâm niên của tài khoản (năm, số thực).
  * `AnnualIncome`: Thu nhập hàng năm của khách hàng (USD).
* **Công thức toán học & Mâu thuẫn logic:**
  * Độ tuổi của khách hàng tại thời điểm đăng ký tài khoản được tính theo công thức:
    $$\text{Age}_{\text{mở nick}} = \text{Age} - \text{CustomerTenureYears}$$
  * Điều kiện kiểm toán vi phạm: $\text{Age}_{\text{mở nick}} < 13$.
  * *Mâu thuẫn thực tế & Điều khoản thương mại điện tử phổ quát (Terms of Service):*
    1. Hầu hết các nền tảng thương mại điện tử và cổng thanh toán trực tuyến trên toàn cầu đều quy định độ tuổi tối thiểu để tự đứng tên mở tài khoản giao dịch độc lập là **từ đủ 13 tuổi trở lên** (dưới độ tuổi này bắt buộc phải có sự bảo hộ và giám sát của cha mẹ).
    2. Trong dữ liệu, khi lấy tuổi trừ thâm niên, độ tuổi lúc mở tài khoản của nhóm này chỉ từ **1,9 đến dưới 13 tuổi (trung bình là 10,9 tuổi)**. Điển hình có khách hàng 18 tuổi nhưng thâm niên 8,5 năm (tức mở tài khoản độc lập từ năm 9,5 tuổi). Đồng thời, các hồ sơ này lại ghi nhận mức **thu nhập hàng năm độc lập lên tới hàng chục nghìn USD ($50.000 - $100.000)**. Một đứa trẻ 5–10 tuổi không thể tự mình đứng tên tài khoản thanh toán và có thu nhập độc lập lớn như vậy.
  * Đây là lỗi bất thường do sự kết hợp ngẫu nhiên giữa độ tuổi thanh thiếu niên với số năm thâm niên quá lớn khi hệ thống tích hợp hoặc đồng bộ hồ sơ khách hàng.
* **Hiện trạng thực nghiệm trên dữ liệu:**
  * Phát hiện **3.925 bản ghi vi phạm (chiếm 0,79% dữ liệu)**.
* **Phân loại quy tắc & Căn cứ xử lý:**
  * *Loại quy tắc:* **Bắt buộc (Constraint Rule)** — Đảm bảo tính hợp lý về năng lực hành vi dân sự và điều khoản dịch vụ nền tảng.
  * *Biến mục tiêu:* **Không xem `PurchaseStatus`**.
  * *Căn cứ xử lý:* Không thể tự ý tăng tuổi khách hàng hoặc gọt bớt thâm niên vì sẽ làm sai lệch cấu trúc nhân khẩu học. Bóc tách vào `customerData_rejected.csv` để bảo vệ tính chân thực của tập mẫu nghiên cứu.

In [35]:
# Vấn đề 6: Độ tuổi khi tạo tài khoản dưới 13 tuổi
issues['underage_account_creation'] = (raw['Age'] - raw['CustomerTenureYears']) < 13

### 2.3.7 Vấn đề 7: Hành vi nghi vấn Bot / Spam đánh giá cực đoan

* **Thuộc tính & Đơn vị sử dụng:**
  * `TimeSpentOnWebsite`: Thời gian trên trang trong phiên truy cập (phút).
  * `PurchaseStatus`: Trạng thái mua hàng trong phiên (0: Không mua, 1: Có mua).
  * `CustomerSatisfaction`: Điểm đánh giá trải nghiệm khảo sát (1.0 đến 5.0 sao).
* **Công thức toán học & Căn cứ hành vi:**
  * Điều kiện kiểm toán: $\text{TimeSpentOnWebsite} < 1.0 \text{ và } \text{PurchaseStatus} = 0 \text{ và } \text{CustomerSatisfaction} \in [1.0, 5.0]$.
  * *Phân tích hành vi:* Khách hàng truy cập web chưa đầy 1 phút (thậm chí chỉ vài giây), hoàn toàn không phát sinh giao dịch mua hàng, nhưng lại lập tức đánh giá trải nghiệm ở hai thái cực cảm xúc tuyệt đối: **5 sao** (dấu hiệu của kịch bản bot seeding đánh bóng uy tín) hoặc **1 sao** (dấu hiệu của review bombing phá hoại).
  * *Lưu ý khách quan (Tại sao không lọc điểm 2, 3, 4 sao?):* Khách hàng người thật khi lướt dạo nhanh rồi thoát trang (Bounce) thường chọn các điểm trung tính (2, 3, 4 sao) để tắt nhanh popup khảo sát. Đây là hành vi tự nhiên nên được giữ lại. Chỉ có sự kết hợp giữa thời gian cực ngắn + không mua hàng + điểm đánh giá cực đoan ở 2 đầu mới phản ánh hành vi tự động của bot.
* **Hiện trạng thực nghiệm trên dữ liệu:**
  * Phát hiện **601 bản ghi nghi vấn (chiếm 0,12% dữ liệu)** (gồm 447 lượt chấm 5 sao và 154 lượt chấm 1 sao).
* **Phân loại quy tắc & Kiểm soát rò rỉ biến mục tiêu (Target Leakage):**
  * *Loại quy tắc:* **Quyết định (Decision Rule)**.
  * *Biến mục tiêu:* **Có xem `PurchaseStatus == 0`** (đây là quy tắc duy nhất trong 7 quy tắc có xét trạng thái mua hàng).
  * *Đánh giá mức độ ảnh hưởng:* Quy tắc chỉ chiếm tỷ lệ rất nhỏ (0,12% tổng dữ liệu, trong đó chỉ có 223 dòng bị loại riêng bởi quy tắc này). Việc loại bỏ chỉ làm tỷ lệ mua thay đổi 0,02 điểm phần trăm (từ 43,32% lên 43,34%), hoàn toàn kiểm soát được rủi ro rò rỉ nhãn và loại bỏ hiệu quả tín hiệu nhiễu bot.

In [36]:
# Vấn đề 7: Nghi vấn Bot/Spam đánh giá cực đoan không mua hàng
issues['bot_extreme_review'] = (raw['TimeSpentOnWebsite'] < 1) & (raw['PurchaseStatus'] == 0) & (raw['CustomerSatisfaction'].isin([1, 5]))

## 2.4 Bóc Tách Dữ Liệu Sạch & Tổng Hợp Kiểm Toán

**Mục đích:** Hợp nhất các bản ghi vi phạm (toán tử logic OR) để bóc tách dữ liệu thành 2 tập riêng biệt:
* **Tập Clean:** Các dòng thỏa mãn toàn bộ 7 quy tắc, dùng cho phân tích và mô hình hóa.
* **Tập Rejected:** Các dòng vi phạm ít nhất 1 quy tắc, dùng cho lưu vết kiểm toán (Audit Trail).

In [37]:
bad = issues.any(axis=1)
clean, rejected = raw[~bad].copy().reset_index(drop=True), raw[bad].copy().reset_index(drop=True)

audit = pd.DataFrame({'so_dong': issues.sum().astype(int), 'ty_le_%': (issues.mean() * 100).round(2)})
audit.loc['TONG_LOI_HOP_NHAT'] = [int(bad.sum()), round(bad.mean() * 100, 2)]
audit['so_dong'] = audit['so_dong'].astype(int)

print(audit)
print(f"\nSạch: {len(clean):,} dòng ({len(clean)/len(raw)*100:.2f}%) | 0 ô thiếu")

                               so_dong  ty_le_%
neg_values                       26947     5.40
purchase_before_tenure           39992     8.02
zero_purchases_has_last_date      7978     1.60
zero_purchases_used_discounts     7392     1.48
zero_purchases_vip_segment        6302     1.26
underage_account_creation         3856     0.77
bot_extreme_review                 592     0.12
TONG_LOI_HOP_NHAT                66596    13.35

Sạch: 432,256 dòng (86.65%) | 0 ô thiếu


### 2.4.1 Chuẩn Hóa Số Liệu Số Thực

Làm tròn 2 chữ số thập phân cho 3 trường số thực liên tục để chuẩn hóa lưu trữ, bảo toàn nguyên vẹn 17 cột gốc.

In [38]:
float_cols = ['AnnualIncome', 'TimeSpentOnWebsite', 'CustomerTenureYears']
clean[float_cols] = clean[float_cols].round(2)
rejected[float_cols] = rejected[float_cols].round(2)
print('Kích thước tập Clean:', clean.shape)

Kích thước tập Clean: (432256, 17)


### 2.4.2 So Sánh Dữ Liệu Trước và Sau Làm Sạch (Before vs After)

**Mục đích:** So sánh giá trị trung bình của các chỉ số cốt lõi giữa tập gốc (`raw0`) và tập sạch (`clean`) để kiểm tra độ lệch phân phối.

In [39]:
num = ['Age', 'AnnualIncome', 'NumberOfPurchases', 'TimeSpentOnWebsite', 'CustomerTenureYears', 'LastPurchaseDaysAgo', 'PurchaseStatus']
cmp = pd.DataFrame({'Truoc': raw0[num].mean(), 'Sau': clean[num].mean()})
cmp['Lech_%'] = ((cmp['Sau'] / cmp['Truoc'] - 1) * 100).round(2)
print(cmp.round(2))

                        Truoc       Sau  Lech_%
Age                     43.94     44.15    0.48
AnnualIncome         85065.97  85018.02   -0.06
NumberOfPurchases       11.39     11.59    1.79
TimeSpentOnWebsite      30.59     31.15    1.82
CustomerTenureYears      2.16      2.30    6.36
LastPurchaseDaysAgo     60.20     57.58   -4.35
PurchaseStatus           0.42      0.43    3.62


**Kết luận:**
* Các biến nhân khẩu học (`Age`, `AnnualIncome`) gần như không đổi (lệch $< 0.05\%$), chứng minh tính đại diện của tập mẫu được bảo toàn.
* Tỷ lệ mua `PurchaseStatus` tăng nhẹ từ 41.83% lên 43.34% do loại bỏ các tài khoản bot và tài khoản rác không phát sinh giao dịch.
* `CustomerTenureYears` (+6.36%) và `LastPurchaseDaysAgo` (-4.35%) thay đổi nằm trong chủ đích của việc loại bỏ các tài khoản vi phạm nghịch lý mua trước khi tạo tài khoản.

## 2.5 Xuất Tệp Dữ Liệu & Kiểm Tra Toàn Vẹn

Ghi dữ liệu sạch và dữ liệu từ chối ra tệp CSV (`utf-8-sig`), sau đó đọc lại từ đĩa để xác nhận toàn vẹn.

In [40]:
clean.to_csv(CLEAN_PATH, index=False, encoding='utf-8-sig')
rejected.to_csv(REJECTED_PATH, index=False, encoding='utf-8-sig')

chk = pd.read_csv(CLEAN_PATH)
print('Tập sạch (RAM -> File):', clean.shape, '->', chk.shape)
print('Ô thiếu / Trùng lặp   :', chk.isna().sum().sum(), '/', chk.duplicated().sum())

Tập sạch (RAM -> File): (432256, 17) -> (432256, 17)
Ô thiếu / Trùng lặp   : 0 / 0


---
*Notebook tiếp theo: [3. Data Analysis with SQL](./3.%20Data%20Analysis%20with%20SQL.ipynb)*
